# Chapter 10 — Probabilistic Models for Classification II: Bayes Classifier, Naive Bayes & Gaussian Mixture Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch10_probabilistic_models_for_classification_ii/ch10_notebook.ipynb)

Companion notebook for **Chapter 10** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- The Bayes classifier picks the class with the highest posterior: likelihood × prior ÷ evidence.
- Priors encode how common each class is. They can change the decision for borderline cases.
- Maximum likelihood estimates parameters. For a Gaussian, that means the sample mean and covariance.
- Use log-likelihoods: products of many small probabilities underflow to zero.
- Naive Bayes assumes independent features (a diagonal covariance). It is simple and fast, but ignores correlations.
- A GMM models multi-clump classes as a weighted sum of Gaussians, trained by the EM algorithm.

## Example 1: Bayes Classifier with Gaussian Densities

We fit one Gaussian per class (mean and covariance) to the adult/child data, then apply Bayes' rule step by step: likelihood, then evidence, then posterior. The likelihoods for the test person are 3.44×10⁻¹⁷ (child) and 5.31×10⁻⁴ (adult), so adult wins overwhelmingly. The second part shows why priors matter. The borderline person (140 cm, 35 kg) is 'Adult' with equal priors but 'Child' once we use the priors of a primary school.

In [1]:
# Chapter 10 - Example 1: Bayes classifier with a Gaussian density for each class
import numpy as np

child = np.array([[98, 16.4], [104, 15.9], [110, 19.8], [115, 19.1], [119, 23.0],
                  [123, 22.4], [127, 27.1], [131, 26.3], [134, 31.0], [101, 17.5]])
adult = np.array([[152, 52.0], [158, 50.5], [161, 63.2], [165, 57.8], [168, 71.0],
                  [172, 64.5], [175, 80.3], [179, 72.9], [183, 88.0], [188, 81.4]])

def gaussian_pdf(x, mu, cov):
    d = len(mu)
    diff = x - mu
    norm = 1.0 / ((2 * np.pi) ** (d / 2) * np.sqrt(np.linalg.det(cov)))
    return norm * np.exp(-0.5 * diff @ np.linalg.inv(cov) @ diff)

# Training: prior + Gaussian parameters (mean, covariance) for each class
N = len(child) + len(adult)
classes = {}
for name, data in (("Child", child), ("Adult", adult)):
    classes[name] = dict(prior=len(data) / N, mu=data.mean(axis=0),
                         cov=np.cov(data, rowvar=False))
    print(f"{name}: prior P = {classes[name]['prior']:.2f}, mean = "
          f"{np.round(classes[name]['mu'], 2)}")

def classify(x, priors=None):
    like = {c: gaussian_pdf(x, p["mu"], p["cov"]) for c, p in classes.items()}
    pri = priors or {c: p["prior"] for c, p in classes.items()}
    evidence = sum(like[c] * pri[c] for c in classes)          # total probability P(x)
    post = {c: like[c] * pri[c] / evidence for c in classes}   # Bayes rule
    return like, post

x = np.array([150, 47.0])
like, post = classify(x)
print(f"\nTest person x = {x}")
for c in classes:
    print(f"  likelihood p(x | {c}) = {like[c]:.2e}   posterior P({c} | x) = {post[c]:.2e}")
print("  Decision:", max(post, key=post.get))

# The prior matters: data collected in a primary school (9 children for every adult)
x2 = np.array([140, 35.0])
for label, pri in (("equal priors", None), ("school priors", {"Child": 0.9, "Adult": 0.1})):
    _, post = classify(x2, pri)
    print(f"\nx2 = {x2} with {label}: P(Child|x) = {post['Child']:.3f}, "
          f"P(Adult|x) = {post['Adult']:.3f} -> {max(post, key=post.get)}")

Child: prior P = 0.50, mean = [116.2   21.85]
Adult: prior P = 0.50, mean = [170.1   68.16]

Test person x = [150.  47.]
  likelihood p(x | Child) = 3.44e-17   posterior P(Child | x) = 6.48e-14
  likelihood p(x | Adult) = 5.31e-04   posterior P(Adult | x) = 1.00e+00
  Decision: Adult

x2 = [140.  35.] with equal priors: P(Child|x) = 0.395, P(Adult|x) = 0.605 -> Adult

x2 = [140.  35.] with school priors: P(Child|x) = 0.855, P(Adult|x) = 0.145 -> Child


## Example 2: Maximum Likelihood and Naive Bayes

First we run a grid search: try thousands of candidate means for the child Gaussian and keep the one with the highest log-likelihood. It lands on the sample mean, which shows that the ML estimate is the sample mean. Then we show why logs are used: the raw likelihood of just 10 points is about 10⁻²⁵. Finally, on the Wine dataset (13 features, 3 classes), naive Bayes uses 26 parameters per class against 104 for a full covariance. It is slightly less accurate here because it ignores correlations.

In [2]:
# Chapter 10 - Example 2: Maximum likelihood estimation, and Naive Bayes vs full Bayes
import numpy as np
from scipy.stats import multivariate_normal
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

child = np.array([[98, 16.4], [104, 15.9], [110, 19.8], [115, 19.1], [119, 23.0],
                  [123, 22.4], [127, 27.1], [131, 26.3], [134, 31.0], [101, 17.5]])
cov = np.cov(child, rowvar=False)

# 1. Search a grid of possible means; keep the one with the highest log-likelihood
best, best_ll = None, -np.inf
for m1 in np.linspace(66.2, 166.2, 201):
    for m2 in np.linspace(-28.15, 71.85, 201):
        ll = multivariate_normal([m1, m2], cov).logpdf(child).sum()  # sum of logs
        if ll > best_ll:
            best, best_ll = (m1, m2), ll
print(f"Grid search best mean : ({best[0]:.2f}, {best[1]:.2f})  log-likelihood {best_ll:.2f}")
print(f"Sample mean (formula) : ({child[:, 0].mean():.2f}, {child[:, 1].mean():.2f})")
print("-> Maximum likelihood estimate of a Gaussian mean = the sample mean.\n")

# 2. Why use logs? The raw likelihood is a product of tiny numbers
p = multivariate_normal(child.mean(0), cov).pdf(child)
print(f"Product of 10 likelihoods: {np.prod(p):.3e}  (sum of logs: {np.log(p).sum():.2f})\n")

# 3. Naive Bayes (diagonal covariance) vs full-covariance Bayes on the Wine dataset
X, y = load_wine(return_X_y=True)              # 178 wines, 13 chemical features, 3 classes
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
d = X.shape[1]
nb = GaussianNB().fit(Xtr, ytr)
full = QuadraticDiscriminantAnalysis(reg_param=0.01).fit(Xtr, ytr)
print(f"Parameters per class: Naive Bayes {2 * d}, full covariance {d + d * (d + 1) // 2}")
print(f"Test accuracy  Naive Bayes: {nb.score(Xte, yte):.3f}   "
      f"Full-covariance Bayes: {full.score(Xte, yte):.3f}")

Grid search best mean : (116.20, 21.85)  log-likelihood -57.67
Sample mean (formula) : (116.20, 21.85)
-> Maximum likelihood estimate of a Gaussian mean = the sample mean.

Product of 10 likelihoods: 9.016e-26  (sum of logs: -57.67)

Parameters per class: Naive Bayes 26, full covariance 104
Test accuracy  Naive Bayes: 0.963   Full-covariance Bayes: 1.000


## Example 3: Gaussian Mixture Model with EM

Class A has two separate clusters. We train a 2-component GMM with EM written out in full. Watch the log-likelihood rise and then stop changing after four iterations, and the learned centers land on the two true clusters at (0, 0) and (4, 4). In the Bayes classifier, a single Gaussian for class A scores 95% while a GMM scores 99%. The final table is the elbow: the log-likelihood jumps from K = 1 to K = 2, then barely improves, so K = 2 is the right choice.

In [3]:
# Chapter 10 - Example 3: Gaussian Mixture Model trained with EM, used in a Bayes classifier
import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import multivariate_normal

rng = np.random.default_rng(0)
# Class A has TWO clusters (multimodal); class B sits in between
A = np.vstack([rng.normal([0, 0], 0.6, (150, 2)), rng.normal([4, 4], 0.6, (150, 2))])
B = rng.normal([2, 2], 0.6, (300, 2))

# --- EM from scratch on class A with K = 2 components
K = 2
mu = A[rng.choice(len(A), K, replace=False)]          # random initial centers
cov = np.array([np.eye(2)] * K)
w = np.full(K, 1 / K)
prev = -np.inf
for it in range(1, 50):
    dens = np.column_stack([w[k] * multivariate_normal(mu[k], cov[k]).pdf(A) for k in range(K)])
    ll = np.log(dens.sum(axis=1)).sum()                # total data log-likelihood
    gamma = dens / dens.sum(axis=1, keepdims=True)     # E-step: responsibilities
    Nk = gamma.sum(axis=0)                             # M-step: re-estimate parameters
    mu = (gamma.T @ A) / Nk[:, None]
    cov = np.array([((gamma[:, k, None] * (A - mu[k])).T @ (A - mu[k])) / Nk[k]
                    for k in range(K)])
    w = Nk / len(A)
    if it <= 4 or abs(ll - prev) < 1e-3:
        print(f"EM iteration {it:2d}: log-likelihood = {ll:9.2f}")
    if abs(ll - prev) < 1e-3:
        print("Converged (change < 0.001).")
        break
    prev = ll
print("Learned centers:", np.round(mu, 2).tolist(), " weights:", np.round(w, 2).tolist())

# --- Bayes classifier: one density per class, single Gaussian vs GMM
X_te = np.vstack([rng.normal([0, 0], 0.6, (100, 2)), rng.normal([4, 4], 0.6, (100, 2)),
                  rng.normal([2, 2], 0.6, (200, 2))])
y_te = np.r_[np.zeros(200), np.ones(200)]
for k in (1, 2):
    gA = GaussianMixture(k, random_state=0).fit(A)
    gB = GaussianMixture(1, random_state=0).fit(B)
    pred = (gB.score_samples(X_te) > gA.score_samples(X_te)).astype(int)  # equal priors
    print(f"Class A modeled with K = {k} component(s): test accuracy {np.mean(pred == y_te):.3f}")

# --- Elbow: total log-likelihood of class A for different K
print("\nK   total log-likelihood")
for k in range(1, 6):
    g = GaussianMixture(k, random_state=0).fit(A)
    print(f"{k}   {g.score(A) * len(A):9.1f}")

EM iteration  1: log-likelihood =  -1140.64
EM iteration  2: log-likelihood =   -749.70
EM iteration  3: log-likelihood =   -746.14
EM iteration  4: log-likelihood =   -746.14
Converged (change < 0.001).
Learned centers: [[-0.1, 0.06], [4.0, 3.99]]  weights: [0.5, 0.5]
Class A modeled with K = 1 component(s): test accuracy 0.950
Class A modeled with K = 2 component(s): test accuracy 0.990

K   total log-likelihood
1     -1012.4
2      -746.1
3      -745.5
4      -743.0
5      -741.8
